In [1]:
pip install ijson pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Data Preprocessing

- Convert the large JSON file (list of nips 2025 papers)
- preprocess → clean text → export to Excel (multiple sheets)

In [8]:
# ===== File Paths =====
# Path to the input JSON file containing ICLR 2025 paper data
input_path = r"C:\Users\ywx1409137\Downloads\2026\1_\ICRI论文分析\nips2025\nips2025.json"

# Path to the output Excel file
output_path = r"C:\Users\ywx1409137\Downloads\2026\1_\ICRI论文分析\nips2025.1.2.xlsx"

In [3]:
# ===== Import Required Libraries =====
import ijson
import pandas as pd
import re
import json

#### Define Excel-Safe Text Cleaning Function
This function removes illegal control characters that are not supported by Excel.

In [15]:
# ===== Text Cleaning Function =====
ILLEGAL_CHARACTERS_RE = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]")

def clean_text(v):
    if isinstance(v, str):
        v = ILLEGAL_CHARACTERS_RE.sub("", v)
        v = v.replace("\n", " ").replace("\r", " ")
        return re.sub(r"\s+", " ", v).strip()
    return v


#### Count the Number of Papers
- This step streams the JSON file and counts how many paper entries (IDs) exist.

In [16]:
with open(input_path, "r", encoding="utf-8") as f:
    papers = json.load(f)

print(f"Total papers: {len(papers)}")

Total papers: 6212


#### Export to Excel

In [17]:
# ===== Extract Fields =====
rows = []

for paper in papers:
    rows.append({
        "id": paper.get("id"),
        "title": clean_text(paper.get("title")),
        "status": paper.get("status"),
        "track": paper.get("track"),
        "primary_area": paper.get("primary_area"),
        "abstract": clean_text(paper.get("abstract")),
        "keywords": clean_text(paper.get("keywords")),
        "author": clean_text(paper.get("author")),
        "author_num": paper.get("author_num"),
        "affiliation": clean_text(paper.get("aff_unique_norm")),
        "country": clean_text(paper.get("aff_country_unique")),
        "rating_avg": paper.get("rating_avg", [None])[0],
        "confidence_avg": paper.get("confidence_avg", [None])[0],
        "gs_citation": paper.get("gs_citation"),
        "openreview": paper.get("openreview"),
        "pdf": paper.get("pdf"),
    })


# ===== Export to Excel =====
df = pd.DataFrame(rows)
df.to_excel(output_path, index=False)

print(f"Export completed: {len(df)} papers saved.")

Export completed: 6212 papers saved.


### Analysis

In [18]:
# %%
# =========================================
# NeurIPS 2025 Company–Research Analysis
# Compact Post-Processing Version
# =========================================

import pandas as pd
import re
from collections import defaultdict
from functools import lru_cache


In [19]:

# =========================================
# Module 0: Load preprocessed data
# =========================================

INPUT_PATH = r"C:\Users\ywx1409137\Downloads\2026\1_\ICRI论文分析\nips2025\nips2025.1.2.xlsx"
OUTPUT_PATH = r"C:\Users\ywx1409137\Downloads\2026\1_\ICRI论文分析\nips2025\nips2025_company_research_analysis.xlsx"

df = pd.read_excel(INPUT_PATH)

# Keep only the fields needed for this analysis
df = df[["id", "status", "affiliation", "country"]].copy()

# Basic safety cleaning
df["id"] = df["id"].astype(str).str.strip()
df["status"] = df["status"].fillna("").astype(str).str.strip()
df["affiliation"] = df["affiliation"].fillna("").astype(str).str.strip()
df["country"] = df["country"].fillna("").astype(str).str.strip()

print(f"Loaded {len(df)} papers")

Loaded 6212 papers


#### Filter Accepted-paper subset

In [ ]:
# =========================================
# Module 1: Accepted-paper subset
# =========================================

ACCEPTED_STATUSES = {"Spotlight", "Poster", "Oral"}

df_accepted = df[df["status"].isin(ACCEPTED_STATUSES)].copy()

print(f"Accepted papers: {len(df_accepted)}")

Accepted papers: 5812


##### Recogine Company and build the dictionary

In [21]:
COMPANY_KEYWORDS = {
    '01ai': ['01.ai'],
    '1x technologies': ['1x technologies'],
    '2077ai': ['2077ai'],
    'abaka': ['abaka'],
    'abb': ['abb'],
    'abbvie': ['abbvie', 'abbvie inc'],
    'adago': ['adago', 'adago gmbh'],
    'adobe': ['adobe'],
    'aeolus labs': ['aeolus labs'],
    'agi': ['agi inc', 'agi, inc'],
    'agibot': ['agibot'],
    'ai sweden': ['ai sweden'],
    'ai2 robotics': ['ai2 robotics', 'ai^2 robotics'],
    'airbus': ['airbus'],
    'aispeech': ['aispeech', 'aispeech ltd'],
    'aithyra': ['aithyra'],
    'algolux': ['algolux'],
    'alibaba': ['alibaba', 'tongyi', 'tongyi lab'],
    'amazon': ['amazon', 'aws'],
    'amd': ['advanced micro devices', 'amd'],
    'anheng technology': ['anheng technology'],
    'ant': ['ant group', 'ant research'],
    'anthropic': ['anthropic'],
    'apart research': ['apart research'],
    'apollo research': ['apollo research'],
    'appier': ['appier', 'appier inc'],
    'apple': ['apple'],
    'aramco': ['aramco', 'aramco houston research center'],
    'arashi vision': ['arashi vision'],
    'archimedes': ['archimedes ai'],
    'armortech': ['armortech'],
    'arquimea': ['arquimea', 'arquimea research'],
    'asapp': ['asapp'],
    'asiainfo': ['asiaInfo', 'asiainfo', 'asiainfo technologies'],
    'ataraxis': ['ataraxis', 'ataraxis.ai'],
    'atcoder': ['atcoder', 'atcoder inc'],
    'augur': ['augur'],
    'autodesk': ['autodesk'],
    'axiomatic': ['axiomatic ai'],
    'baichuan': ['baichuan', 'baichuan inc'],
    'baidu': ['baidu'],
    'bbc': ['bbc', 'british broadcasting corporation'],
    'beijing humanoid robot': ['beijing humanoid robot',
                                'beijing humanoid robot innovation center',
                                'x-humanoid'],
    'beijing xyz embodied': ['beijing xyz embodied ai'],
    'biogen': ['biogen', 'biogen inc'],
    'biomap': ['biomap', 'biomap research'],
    'boeing': ['boeing', 'boeing research and technology'],
    'booking': ['booking ai', 'booking ai research'],
    'booz allen hamilton': ['booz allen hamilton'],
    'borealis': ['borealis ai'],
    'bosch': ['bosch'],
    'boson': ['boson ai'],
    'boston consulting': ['bcg', 'boston consulting group'],
    'brainchip': ['brainchip', 'brainchip inc'],
    'bytedance': ['bytedance', 'tiktok'],
    'calico': ['calico', 'calico labs'],
    'canon': ['canon', 'canon inc'],
    'capital one': ['capital one'],
    'cardiio': ['cardiio', 'cardiio inc'],
    'cephia': ['cephia ai'],
    'cerebras': ['cerebras', 'cerebras systems'],
    'cgi': ['cgi'],
    'chai discovery': ['chai discovery'],
    'china communications it': ['china communications information technology '
                                'group',
                                'china communications it'],
    'china electronics': ['china electronics',
                        'china electronics technology group corporation'],
    'china international capital': ['china international capital',
                                    'china international capital corporation'],
    'china mobile': ['china mobile', 'china mobile communications group'],
    'china mobile fintech': ['china mobile financial technology',
                            'china mobile fintech'],
    'china railway': ['china railway', 'china railway xi’an bureau'],
    'china telecom': ['china telecom'],
    'china unicom': ['china unicom', 'china unicom network communications'],
    'china unicom global': ['china unicom global'],
    'cimda': ['cimda'],
    'cisco': ['cisco'],
    'cisdi': ['cisdi', 'cisdi group'],
    'cnnc': ['cnnc'],
    'cohere': ['cohere'],
    'collinear': ['collinear ai'],
    'collov': ['collov', 'collov.ai'],
    'commonground': ['commonground'],
    'contextual': ['contextual ai'],
    'contramont research': ['contramont research'],
    'corsound': ['corsound ai'],
    'cruise': ['cruise', 'gm cruise'],
    'cuspi': ['cuspi', 'cuspi ai'],
    'cvte': ['cvte', 'cvte research institute'],
    'cyagen biosciences': ['cyagen biosciences'],
    'cyberagent': ['cyberagent'],
    'cycraft': ['cycraft', 'cycraft technology corporation'],
    'databricks': ['databricks'],
    'datacanvas': ['datacanvas'],
    'datadog': ['datadog'],
    'dayhoff labs': ['dayhoff labs'],
    'dbappsecurity': ['dbappsecurity', 'dbappsecurity co., ltd'],
    'deepautoai': ['deepauto.ai'],
    'deepglint': ['deepglint'],
    'deeproute': ['deeproute', 'deeproute.ai'],
    'deepscenario': ['deepscenario', 'deepscenario gmbh'],
    'dell': ['dell'],
    'denso': ['denso', 'denso corporation', 'denso it laboratory'],
    'dexforce': ['dexforce', 'dexforce co. ltd'],
    'didi': ['didi', 'didi research'],
    'didi international': ['didi international',
                            'didi international business group'],
    'disney': ['disney'],
    'dmatrix': ['d-matrix', 'd-matrix corporation'],
    'dolby': ['dolby', 'dolby laboratories'],
    'duxiaoman': ['du xiaoman',
                'du xiaoman technology',
                'duxiaoman',
                'duxiaoman technology',
                'dxm'],
    'earth species project': ['earth species project'],
    'ebay': ['ebay', 'ebay inc'],
    'ebtech': ['beijing eb technology', 'ebtech'],
    'edf': ['edf', 'electricite de france'],
    'eigent': ['eigent ai', 'eigent.ai'],
    'eleutherai': ['eleutherai'],
    'eli lilly': ['eli lilly'],
    'ellison': ['ellison institute', 'ellison institute for technology'],
    'emergence': ['emergence ai'],
    'emmi': ['emmi ai'],
    'entos therapeutics': ['entos',
                            'entos inc',
                            'entos therapeutics',
                            'entos, inc'],
    'entrust': ['entrust', 'entrust corp'],
    'exawizards': ['exawizards', 'exawizards inc'],
    'farai': ['far ai', 'far.ai'],
    'featurespace': ['featurespace', 'featurespace ltd'],
    'feedzai': ['feedzai'],
    'field': ['field ai'],
    'fin': ['fin ai'],
    'finsyth': ['finsyth ai'],
    'flawless': ['flawless ai'],
    'floma': ['floma', 'floma inc'],
    'flower labs': ['flower labs'],
    'ford': ['ford', 'ford corporation'],
    'forest': ['forest ai'],
    'foxconn': ['foxconn', 'hon hai technology group'],
    'fpt software': ['fpt software'],
    'fractal analytics': ['fractal ai', 'fractal analytics'],
    'fujitsu': ['fujitsu'],
    'futurehouse': ['futurehouse'],
    'galbot': ['galbot'],
    'ganfried research': ['ganfried research', 'ganzfried research'],
    'gauss labs': ['gauss labs'],
    'ge healthcare': ['ge healthcare'],
    'genentech': ['genentech'],
    'generalist': ['generalist ai'],
    'gigaai': ['gigaai'],
    'glam': ['glam ai'],
    'global energy interconnection': ['global energy interconnection',
                                    'global energy interconnection research '
                                    'institute'],
    'goodfire': ['goodfire ai'],
    'google': ['alphabet', 'deepmind', 'google'],
    'grabotics': ['grabotics', 'grabotics inc'],
    'gray swan': ['gray swan', 'gray swan ai'],
    'green dynamics': ['green dynamics', 'greendynamics'],
    'grg banking': ['grg banking', 'grg banking equipment'],
    'gridmatic': ['gridmatic'],
    'gyges labs': ['gyges labs'],
    'hangzhou yingshen': ['hangzhou yingshen',
                        'hangzhou yingshen intelligent technology'],
    'hangzhou yufan': ['hangzhou yufan', 'hangzhou yufan intelligent technology'],
    'haomoai': ['haomo ai', 'haomo.ai'],
    'hiddenlayer': ['hiddenlayer'],
    'hifx': ['hifx', 'hifx it & media services'],
    'hikvision': ['hikvision'],
    'hiregit': ['hiregit'],
    'hitachi': ['hitachi'],
    'hithink': ['hithink',
                'hithink royalflush',
                'hithink royalflush information network'],
    'honda': ['honda', 'honda r&d'],
    'horizon robotics': ['horizon robotics'],
    'houmo': ['houmo', 'houmo ai', 'houmo.ai'],
    'hp': ['hewlett packard', 'hp'],
    'hp enterprise': ['hewlett packard enterprise', 'hp enterprise', 'hpe'],
    'htx': ['htx'],
    'huawei': ['huawei'],
    'hugging face': ['hugging face'],
    'humane intelligence': ['humane intelligence'],
    'humanloop': ['humanloop', 'humanloop inc'],
    'iambic therapeutics': ['iambic therapeutics'],
    'ibm': ['ibm', 'international business machines'],
    'idealism': ['idealism', 'idealism beijing technology'],
    'ihuman': ['ihuman', 'ihuman corporation'],
    'indeed': ['indeed'],
    'inephany': ['inephany', 'inephany ltd'],
    'infinigence': ['infinigence ai'],
    'infix': ['infix ai', 'infix.ai'],
    'insta360': ['insta360', 'insta360 innovation technology'],
    'intel': ['intel'],
    'intellect frontier': ['intellect frontier'],
    'intellifusion': ['intellifusion'],
    'intuit': ['intuit', 'intuit ai'],
    'irt systemx': ['irt systemx', 'systemx'],
    'isomorphic labs': ['isomorphic labs'],
    'japan broadcastingoration': ['japan broadcasting corporation'],
    'jd': ['jd', 'jingdong'],
    'jetbrains': ['jetbrains'],
    'jiiov technology': ['jiiov technology'],
    'jina': ['jina ai'],
    'jpmorgan chase': ['j.p. morgan', 'j.p. morgan chase', 'jpmorgan chase'],
    'jrbrown bio': ['jrbrown bio', 'jrbrown bio consulting'],
    'kausable': ['kausable', 'kausable gmbh'],
    'kensho': ['kensho', 'kensho technologies'],
    'keybyte': ['keybyte', 'keybyte llc'],
    'knowin': ['knowin ai'],
    'krafton': ['krafton', 'krafton inc'],
    'kuaishou technology': ['kuaishou technology'],
    'kumo': ['kumo ai'],
    'kunbyte': ['kunbyte'],
    'kunlun skywork': ['kunlun skywork', 'kunlun skywork ai'],
    'lambda labs': ['lambda labs'],
    'lasr labs': ['lasr labs'],
    'layer6': ['layer 6 ai', 'layer6 ai'],
    'learnable': ['learnable', 'learnable, inc'],
    'leeroo': ['leeroo', 'leeroo ai'],
    'lenovo': ['lenovo'],
    'lg': ['lg'],
    'lg electronics': ['lg electronics', 'lg electronics canada'],
    'li auto': ['li auto', 'li auto inc'],
    'light robotics': ['light robotics'],
    'lila sciences': ['lila sciences'],
    'linkedin': ['linkedin', 'linkedin corporation'],
    'lloyds banking': ['lloyds banking group'],
    'lockheed martin': ['lockheed martin', 'lockheed martin corporation'],
    'lovart': ['lovart ai'],
    'luma': ['luma ai'],
    'lyoration': ['ly corporation'],
    'magicly': ['magicly ai'],
    'malou tech': ['malou tech', 'malou tech inc'],
    'mantra': ['mantra', 'mantra inc'],
    'matta labs': ['matta labs'],
    'maumai': ['maum.ai'],
    'mediatek': ['mediatek', 'mediatek inc'],
    'megagon labs': ['megagon labs'],
    'megvii': ['megvii'],
    'meitu': ['meitu'],
    'meituan': ['meituan'],
    'memtensor': ['memtensor', 'memtensor (shanghai) technology'],
    'mercator ocean': ['mercator ocean', 'mercator ocean international'],
    'meshcapade': ['meshcapade'],
    'meta': ['facebook', 'meta', 'meta rl', 'meta rl research'],
    'metacarbon': ['metacarbon'],
    'metax': ['metaX',
            'metax',
            'metax integrated circuit',
            'metax integrated circuit (shanghai)'],
    'metr': ['metr'],
    'microsoft': ['microsoft', 'msft'],
    'midea': ['midea', 'midea group'],
    'minimax': ['minimax'],
    'mistral': ['mistral ai'],
    'mitsubishi electric': ['mitsubishi electric',
                            'mitsubishi electric research laboratories'],
    'mizuhodl fintech': ['mizuho-dl financial technology co., ltd',
                        'mizuho-dl fintech',
                        'mizuho–dl financial technology'],
    'mlcommons': ['mlcommons'],
    'mongodb': ['mongodb'],
    'monta': ['monta ai'],
    'moonshot': ['moonshot ai'],
    'motor': ['motor ai'],
    'mts': ['mts ai'],
    'mujin': ['mujin', 'mujin, inc'],
    'multion': ['multion', 'multion inc'],
    'navana tech': ['navana tech',
                    'navana tech india private limited',
                    'navana.ai',
                    'navanatech'],
    'naver': ['naver', 'naver corporation'],
    'near earth autonomy': ['near earth autonomy'],
    'neb ius': ['neb ius ai', 'nebius ai'],
    'nec': ['nec', 'nec corporation'],
    'netease': ['netease', 'netease inc'],
    'netflix': ['netflix'],
    'netmind': ['netmind'],
    'newstrong': ['newstrong', 'sichuan newstrong uhd video technology'],
    'nexusflow': ['nexusflow', 'nexusflow.ai'],
    'nnaisense': ['nnaisense'],
    'nokia': ['nokia', 'nokia bell labs'],
    'northwest research associates': ['northwest research associates',
                                    'northwest research associates, inc'],
    'novator energy': ['novator energy', 'novator energy inc'],
    'nsfocus': ['nsfocus'],
    'ntt': ['ntt', 'ntt research'],
    'nubank': ['nubank'],
    'nuro': ['nuro', 'nuro.ai'],
    'nvidia': ['nvidia'],
    'nxai': ['nxai'],
    'octanove labs': ['octanove labs'],
    'omelet': ['omelet', 'omelet, inc'],
    'onekq lab': ['onekq lab'],
    'onto innovation': ['onto innovation'],
    'ontocord': ['ontocord', 'ontocord.ai'],
    'openai': ['openai'],
    'opentech edv': ['opentech edv'],
    'oppo': ['oppo', 'oppo mobile telecommunications'],
    'oracle': ['oracle', 'oracle health ai'],
    'origin wireless': ['origin wireless'],
    'palantir': ['palantir'],
    'panasonic': ['panasonic'],
    'parameter lab': ['parameter lab'],
    'petuum': ['petuum', 'petuum inc'],
    'physical intelligence': ['physical intelligence'],
    'pickford': ['pickford ai'],
    'pinecone': ['pinecone', 'pinecone systems'],
    'pluralis': ['pluralis', 'pluralis ai', 'pluralis research'],
    'polyshape': ['polyshape'],
    'ponyai': ['pony ai', 'pony.ai'],
    'poolside': ['poolside'],
    'precur': ['precur ai'],
    'prediction guard': ['prediction guard'],
    'preferred networks': ['preferred networks', 'preferred networks inc'],
    'presight': ['presight'],
    'prior labs': ['prior labs'],
    'procter and gamble': ['p&g', 'procter & gamble', 'procter and gamble'],
    'profluent': ['profluent', 'profluent bio', 'profluent bio inc'],
    'protagolabs': ['protagolabs'],
    'provisio': ['provisio', 'provisio gmbh'],
    'proxima fusion': ['proxima fusion', 'proxima fusion gmbh'],
    'pruna': ['pruna ai'],
    'qihoo 360': ['qihoo 360'],
    'qualcomm': ['qualcomm'],
    'quansight': ['quansight', 'quansight labs'],
    'quantstamp': ['quantstamp'],
    'radiusai': ['radiusai', 'radiusai inc'],
    'rakuten': ['rakuten', 'rakuten asia'],
    'rawmantic': ['rawmantic ai'],
    'rebuilderai': ['rebuilderai'],
    'red dragon': ['red dragon ai'],
    'red hat': ['red hat', 'red hat inc'],
    'redwood research': ['redwood research'],
    'reka': ['reka ai'],
    'relational': ['relational ai', 'relationalai'],
    'respiq': ['respiq'],
    'roblox': ['roblox', 'roblox corporation'],
    'roboflow': ['roboflow', 'roboflow, inc'],
    'roche': ['f. hoffmann-la roche', 'roche'],
    'rtx': ['rtx'],
    's2t': ['s2t ai', 's2t.ai'],
    'safran': ['safran', 'safran group'],
    'sakana': ['sakana ai'],
    'salesforce': ['salesforce'],
    'samsung': ['samsung', 'samsung semiconductor', 'samsung semiconductor inc'],
    'scale': ['scale ai'],
    'scite': ['scite', 'scite.ai'],
    'se3labs': ['se3-labs'],
    'sea': ['sea ai lab', 'sea group'],
    'securebio': ['securebio'],
    'sedai': ['sedai', 'sedai inc'],
    'seetacloud': ['nanjing seetacloud technology', 'seetacloud'],
    'sensetime': ['sensetime'],
    'sentinel4d': ['sentinal4d', 'sentinel4d'],
    'servicenow': ['servicenow'],
    'ses': ['ses ai'],
    'sf technology': ['sf technology', 'sf-technology'],
    'shanda': ['shanda', 'shanda group'],
    'shanghai yincheng intelligent': ['shanghai yincheng intelligent'],
    'shanshu': ['shanshu', 'shanshu ai', 'shanshu.ai'],
    'shengshu technology': ['shengshu technology'],
    'shenzhen microbt': ['shenzhen microbt',
                        'shenzhen microbt electronics technology'],
    'si analytics': ['si analytics'],
    'signal1': ['signal 1 ai', 'signal1 ai'],
    'simplatform': ['simplatform', 'simplatform co. ltd'],
    'simular': ['simular'],
    'sk shieldus': ['sk shieldus'],
    'skild': ['skild ai'],
    'skywork': ['skywork ai'],
    'snowflake': ['snowflake', 'snowflake inc'],
    'softbank': ['softbank', 'softbank group'],
    'solenix': ['solenix', 'solenix engineering', 'solenix engineering gmbh'],
    'sony': ['sony', 'sony ai', 'sony corporation', 'sony europe'],
    'sorbus': ['sorbus ai'],
    'spaitial': ['spaitial', 'spaitial ltd'],
    'spector': ['spector', 'spector inc'],
    'speechmatics': ['speechmatics'],
    'squirrel': ['squirrel ai'],
    'stability': ['stability', 'stability ai'],
    'synopsys': ['synopsys', 'synopsys inc'],
    'synthlabs': ['synthlabs'],
    'tabularis': ['tabularis', 'tabularis.ai'],
    'tal education': ['tal', 'tal education', 'tal education group'],
    'tapall': ['tapall', 'tapall.ai'],
    'taptap': ['taptap'],
    'tech innovation': ['tech innovation institute',
                        'technology innovation institute'],
    'tencent': ['tencent', 'tencent technology', 'wechat', 'wechat ai lab'],
    'teraflop': ['teraflop ai'],
    'thales': ['thales', 'thales group'],
    'theori': ['theori'],
    'thirdai': ['thirdai', 'thirdai corp'],
    'thoughtworks': ['thoughtworks'],
    'tipplyai': ['tipply ai', 'tipplyai'],
    'together': ['together ai'],
    'topaz labs': ['topaz labs'],
    'toshiba': ['toshiba', 'toshiba corporation'],
    'toyota': ['toyota',
                'toyota motor corporation',
                'toyota motors europe',
                'toyota research institute'],
    'transperfect': ['transperfect'],
    'trend micro': ['trend micro', 'trend micro inc'],
    'truthful': ['truthful ai'],
    'tusimple': ['tusimple'],
    'twenty billion neurons': ['twenty billion neurons'],
    'uber': ['uber', 'uber technologies'],
    'uls': ['ul research institutes'],
    'united automotive electronic systems': ['united automotive electronic '
                                            'systems'],
    'upwork': ['upwork', 'upwork global', 'upwork global inc'],
    'valence labs': ['valence labs'],
    'valeo': ['valeo'],
    'vaneval': ['vaneval'],
    'veraitech': ['veraitech'],
    'veris': ['veris ai'],
    'vinai': ['vinai', 'vinai research'],
    'vinci4d': ['vinci4d', 'vinci4d.ai'],
    'visa': ['visa', 'visa inc'],
    'visaze': ['visaze', 'visaze llc'],
    'vivo': ['vivo', 'vivo mobile communication'],
    'volkswagen': ['cariad se', 'volkswagen', 'volkswagen ag'],
    'voyage': ['voyage ai'],
    'walmart': ['walmart', 'walmart inc'],
    'waymo': ['waymo'],
    'wayve': ['wayve'],
    'weaviate': ['weaviate'],
    'webank': ['webank'],
    'windborne systems': ['windborne systems'],
    'work': ['work'],
    'workato': ['workato'],
    'xai': ['xai'],
    'xaira therapeutics': ['xaira therapeutics'],
    'xiaohongshu': ['xiaohongshu'],
    'xiaomi': ['xiaomi', 'xiaomi corporation'],
    'xtalpi': ['xtalpi', 'xtalpi inc'],
    'yandex': ['yandex'],
    'yantronic': ['yantronic', '言创智信'],
    'yellow technologies': ['yellow technologies'],
    'yimu': ['yimu'],
    'yition technology': ['yition technology', 'yition technology co., ltd'],
    'yixiaomo': ['yixiaomo', 'yixiaomo.inc'],
    'yuanshi technology': ['yuanshi technology'],
    'zuoyebang': ['zuoyebang', 'zuoyebang education technology'],
}

##### Research-organization Dictionary

In [22]:
RESEARCH_KEYWORDS = [
    "university", "college",
    "institute", "laboratory", "lab",
    "research center", "research centre",
    "academy",

    # Common public / independent research organizations
    "max planck", "fraunhofer", "inria", "cnrs",
    "dfki", "cifar", "mila", "vector institute",
    "allen institute", "janelia", "helmholtz", "riken",
]

#### Region definitions

In [ ]:
EUROPE = {
    "Albania", "Austria", "Belgium", "Bulgaria", "Croatia",
    "Cyprus", "Czech Republic", "Denmark", "Estonia",
    "Finland", "France", "Germany", "Greece", "Hungary",
    "Ireland", "Italy", "Latvia", "Liechtenstein",
    "Lithuania", "Luxembourg", "Netherlands", "Norway",
    "Poland", "Portugal", "Romania", "Serbia", "Slovakia",
    "Slovenia", "Spain", "Sweden", "Switzerland",
    "United Kingdom", "Ukraine", "Israel",
}

NORTH_AMERICA = {
    "United States", "Canada", "Puerto Rico"
}

ASIA = {
    "Armenia", "Bangladesh", "Brunei Darussalam", "China",
    "Georgia", "Hong Kong", "India", "Indonesia", "Iran", "Iraq",
    "Japan", "Lebanon", "Macau", "Malaysia", "Nepal",
    "Pakistan", "Philippines", "Qatar", "Saudi Arabia",
    "Singapore", "South Korea", "Taiwan", "Thailand", "Türkiye",
    "United Arab Emirates", "Vietnam"
}

LATIN_AMERICA = {
    "Argentina", "Brazil", "Chile", "Colombia",
    "Mexico", "Peru", "Uruguay"
}

AFRICA = {
    "Egypt", "Morocco", "Nigeria", "Rwanda",
    "South Africa", "Uganda"
}

OCEANIA = {
    "Australia", "New Zealand"
}

##### Utility functions

In [25]:
# =========================================
# Matching logic
# =========================================
def split_affiliations(text: str):
    """
    Split a full affiliation string into smaller units.
    This is a lightweight heuristic version intended for notebook analysis.
    """
    if not isinstance(text, str):
        return []
    parts = re.split(r";|/|\|| and ", text)
    return [p.strip() for p in parts if p.strip()]


def match_company_safe(affiliation: str, keywords):
    """
    Return True if any company keyword appears in the affiliation string.
    """
    if not isinstance(affiliation, str):
        return False
    aff = affiliation.lower()
    return any(kw.lower() in aff for kw in keywords)


def extract_companies_from_affiliation(affiliation: str):
    """
    Extract all matched companies from one affiliation string.
    """
    aff = affiliation.lower()
    hits = set()

    for company, keywords in COMPANY_KEYWORDS.items():
        if any(kw.lower() in aff for kw in keywords):
            hits.add(company)

    return hits


def has_research_partner(affiliation: str):
    """
    A paper is treated as company–research collaboration if
    the affiliation string contains at least one company signal
    and at least one academic / research-organization signal.
    """
    if not isinstance(affiliation, str):
        return False

    aff = affiliation.lower()

    has_company = any(
        kw.lower() in aff
        for kws in COMPANY_KEYWORDS.values()
        for kw in kws
    )

    has_research = any(
        kw.lower() in aff for kw in RESEARCH_KEYWORDS
    )

    return has_company and has_research


def count_regions(country_text: str):
    """
    Count whether a paper touches each region based on its country field.
    This is paper-level counting, not institution-level geocoding.
    """
    text = str(country_text)

    return {
        "Europe Research": int(any(c in text for c in EUROPE)),
        "North America Research": int(any(c in text for c in NORTH_AMERICA)),
        "Asia Research": int(any(c in text for c in ASIA)),
        "Latin America Research": int(any(c in text for c in LATIN_AMERICA)),
        "Africa Research": int(any(c in text for c in AFRICA)),
        "Oceania Research": int(any(c in text for c in OCEANIA)),
    }

#### Export Excel

In [26]:
# =========================================
# Sheet 1 / Sheet 2
# Company summary for all papers and accepted papers
# =========================================

def build_company_summary(input_df: pd.DataFrame) -> pd.DataFrame:
    """
    Build company-level summary table.

    Output columns:
    - Company
    - Total Papers
    - Papers with Research Orgs
    - Europe Research
    - North America Research
    - Asia Research
    - Latin America Research
    - Africa Research
    - Oceania Research
    - All_Paper_IDs
    - Research_Paper_IDs
    """
    results = []

    for company, keywords in COMPANY_KEYWORDS.items():
        company_df = input_df[
            input_df["affiliation"].apply(lambda x: match_company_safe(x, keywords))
        ].copy()

        if company_df.empty:
            continue

        company_df["has_research_partner"] = company_df["affiliation"].apply(
            has_research_partner
        )

        company_with_research = company_df[company_df["has_research_partner"]].copy()

        region_totals = {
            "Europe Research": 0,
            "North America Research": 0,
            "Asia Research": 0,
            "Latin America Research": 0,
            "Africa Research": 0,
            "Oceania Research": 0,
        }

        for _, row in company_with_research.iterrows():
            region_flags = count_regions(row["country"])
            for k, v in region_flags.items():
                region_totals[k] += v

        results.append({
            "Company": company,
            "Total Papers": len(company_df),
            "Papers with Research Orgs": len(company_with_research),
            "Europe Research": region_totals["Europe Research"],
            "North America Research": region_totals["North America Research"],
            "Asia Research": region_totals["Asia Research"],
            "Latin America Research": region_totals["Latin America Research"],
            "Africa Research": region_totals["Africa Research"],
            "Oceania Research": region_totals["Oceania Research"],
            "All_Paper_IDs": company_df["id"].tolist(),
            "Research_Paper_IDs": company_with_research["id"].tolist(),
        })

    if not results:
        return pd.DataFrame(columns=[
            "Company", "Total Papers", "Papers with Research Orgs",
            "Europe Research", "North America Research", "Asia Research",
            "Latin America Research", "Africa Research", "Oceania Research",
            "All_Paper_IDs", "Research_Paper_IDs"
        ])

    return (
        pd.DataFrame(results)
        .sort_values(by="Total Papers", ascending=False)
        .reset_index(drop=True)
    )


sheet_all = build_company_summary(df)
sheet_accepted = build_company_summary(df_accepted)

print("Sheet 1 rows:", len(sheet_all))
print("Sheet 2 rows:", len(sheet_accepted))

Sheet 1 rows: 395
Sheet 2 rows: 376


In [27]:
# =========================================
# Sheet 3
# Company submission / acceptance summary
# =========================================

acceptance_rows = []

for company, keywords in COMPANY_KEYWORDS.items():
    company_all = df[
        df["affiliation"].apply(lambda x: match_company_safe(x, keywords))
    ]
    company_acc = df_accepted[
        df_accepted["affiliation"].apply(lambda x: match_company_safe(x, keywords))
    ]

    all_count = len(company_all)
    acc_count = len(company_acc)
    not_acc_count = all_count - acc_count

    acceptance_rate = round(acc_count / all_count * 100, 1) if all_count > 0 else 0.0

    if all_count == 0:
        continue

    acceptance_rows.append({
        "Company": company,
        "All Submissions": all_count,
        "Accepted Papers": acc_count,
        "Not Accepted Papers": not_acc_count,
        "Acceptance Rate (%)": acceptance_rate,
    })

sheet_acceptance = (
    pd.DataFrame(acceptance_rows)
    .sort_values(by="All Submissions", ascending=False)
    .reset_index(drop=True)
)

print("Sheet 3 rows:", len(sheet_acceptance))

Sheet 3 rows: 395


In [28]:
# =========================================
# Sheet 4
# Research organization collaboration table
# =========================================

# This table is built on accepted papers only, which is usually cleaner
# for presenting final collaboration outputs.

# Placeholder organization-country dictionary.
# Fill this only when you want more accurate institution geocoding.
ORG_COUNTRY_MAP = {
    # Example:
    # "Tsinghua University": "China (Mainland)",
    # "Peking University": "China (Mainland)",
    # "Massachusetts Institute of Technology": "United States",
}


def normalize_org_text(text: str) -> str:
    """
    Lightweight normalization for institution names.
    """
    text = str(text).strip()
    text = text.strip(";")
    text = re.sub(r"\s+", " ", text)
    text = text.replace("–", "-")
    return text


def is_research_org(part: str) -> bool:
    """
    Detect whether one affiliation segment looks like an academic
    or public research institution.
    """
    if not isinstance(part, str):
        return False

    p = part.lower()
    return any(k.lower() in p for k in RESEARCH_KEYWORDS)


def map_org_to_country(org_name: str, fallback_country_text: str = "") -> str:
    """
    Map a research organization to a country.
    First use the manual dictionary.
    If missing, fall back to the paper-level country field when possible.
    """
    org_name = normalize_org_text(org_name)

    if org_name in ORG_COUNTRY_MAP:
        return ORG_COUNTRY_MAP[org_name]

    fallback_country_text = str(fallback_country_text)

    # Simple fallback using the paper country string
    # You can refine this later if needed
    for c in EUROPE | NORTH_AMERICA | ASIA | LATIN_AMERICA | AFRICA | OCEANIA:
        if c in fallback_country_text:
            return c

    return "Unknown"


def country_to_region(country: str) -> str:
    """
    Convert country to region label.
    """
    if country in EUROPE:
        return "Europe"
    if country in NORTH_AMERICA:
        return "North America"
    if country in ASIA:
        return "Asia"
    if country in LATIN_AMERICA:
        return "Latin America"
    if country in AFRICA:
        return "Africa"
    if country in OCEANIA:
        return "Oceania"
    return "Unknown"


research_org_stats = defaultdict(lambda: {
    "paper_ids": set(),
    "companies": defaultdict(int),
    "country": "Unknown",
    "region": "Unknown",
})

for _, row in df_accepted.iterrows():
    paper_id = row["id"]
    affiliation = row["affiliation"]
    country_text = row["country"]

    parts = split_affiliations(affiliation)

    companies_in_paper = set()
    research_orgs_in_paper = set()

    for p in parts:
        matched_companies = extract_companies_from_affiliation(p)
        if matched_companies:
            companies_in_paper.update(matched_companies)
            continue

        if is_research_org(p):
            research_orgs_in_paper.add(normalize_org_text(p))

    if not companies_in_paper or not research_orgs_in_paper:
        continue

    for org in research_orgs_in_paper:
        research_org_stats[org]["paper_ids"].add(paper_id)

        if research_org_stats[org]["country"] == "Unknown":
            mapped_country = map_org_to_country(org, country_text)
            research_org_stats[org]["country"] = mapped_country
            research_org_stats[org]["region"] = country_to_region(mapped_country)

        for company in companies_in_paper:
            research_org_stats[org]["companies"][company] += 1


research_rows = []

for org, info in research_org_stats.items():
    research_rows.append({
        "Research Org": org,
        "Country": info["country"],
        "Region": info["region"],
        "Paper Appearances (Institution)": len(info["paper_ids"]),
        "Company Count": len(info["companies"]),
        "Partner Companies": "; ".join(
            f"{c} ({n})"
            for c, n in sorted(
                info["companies"].items(),
                key=lambda x: x[1],
                reverse=True
            )
        ),
        "Paper IDs": sorted(info["paper_ids"]),
    })

sheet_research_orgs = (
    pd.DataFrame(research_rows)
    .sort_values(by="Paper Appearances (Institution)", ascending=False)
    .reset_index(drop=True)
)

print("Sheet 4 rows:", len(sheet_research_orgs))

Sheet 4 rows: 702


In [29]:
# =========================================
# Export all outputs to one Excel file
# =========================================

with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    sheet_all.to_excel(writer, sheet_name="1_all_submissions", index=False)
    sheet_accepted.to_excel(writer, sheet_name="2_accepted_only", index=False)
    sheet_acceptance.to_excel(writer, sheet_name="3_acceptance_summary", index=False)
    sheet_research_orgs.to_excel(writer, sheet_name="4_research_orgs", index=False)

print("✅ Analysis finished")
print(f"📄 Output saved to: {OUTPUT_PATH}")

✅ Analysis finished
📄 Output saved to: C:\Users\ywx1409137\Downloads\2026\1_\ICRI论文分析\nips2025\nips2025_company_research_analysis.xlsx
